# cell-akamai · Lane A · EXP-TOKEN-KINETICS-001

**Environment:** hosted Akamai JupyterLab, CPU-only client notebook with GPU-backed vLLM service. Run workshop `00_connect_and_verify.ipynb` first. **This notebook is a live experiment**, not a simulated dataset.

**Measurements:** actual returned completion token count, streamed chunk event times, raw `/metrics` snapshots, request-level tokens/s and inter-request tokens/s². **Not measured:** model-specific per-token timestamps, true server TTFT, biological accuracy, Anticube, ΔG*, portfolio MMR or signing.

In [ ]:
from pathlib import Path
import os, sys, json
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "cell_akamai").is_dir() else HERE.parent
assert (ROOT / "cell_akamai").is_dir(), "Unzip the Lane A package and open this notebook from its notebooks/ directory"
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Serving URL:", os.getenv("VLLM_HOST", "http://vllm:8000/v1"))
print("Served model:", os.getenv("MODEL_NAME", "Qwen/Qwen3-4B"))
print("Namespace:", os.getenv("NAMESPACE", "UNKNOWN"))
# Do NOT display VLLM_API_KEY, cookies or KUBECONFIG contents.

## A. Read-only connection verification

No GPU deployment changes. The model returned by `/models` is the **served** model. Changing `MODEL_NAME` does not change what the GPU has loaded.

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=os.getenv("VLLM_HOST", "http://vllm:8000/v1"),
                api_key=os.getenv("VLLM_API_KEY", "not-needed"), timeout=120)
models = [m.id for m in client.models.list().data]
print("Server-reported models:", models)
model = os.getenv("MODEL_NAME", "Qwen/Qwen3-4B")
assert model in models, f"Requested {model!r} not in served IDs: {models}"

## B. Live collection (four sequential requests)

We capture content **hashes only**, not original prompt/output strings in raw output files. The prompt identity is pinned to its content hash. The experiment samples `/metrics` on a read-only background thread every second; failures are recorded as FAILED rather than silently replaced.

In [ ]:
from cell_akamai.collect import collect_experiment
from cell_akamai.replay import verify_run
from cell_akamai.kinetics import bytes_digest

run_directory = collect_experiment(
    base_url=os.getenv("VLLM_HOST", "http://vllm:8000/v1"),
    model=model,
    run_count=4,
    max_tokens=128,
    metric_interval_s=1.0,
    prompt_ref="aging_three_mechanisms_v1",
    model_revision=os.getenv("MODEL_REVISION", "UNKNOWN"),
    output_root=ROOT / "runs",
    source_class="OBSERVED",
)
print("Run directory:", run_directory)
print("Readback:", verify_run(run_directory))

## C. Review raw counts and deterministic replay

In [ ]:
from cell_akamai.replay import load_jsonl
rows = load_jsonl(run_directory / "inference_observations.jsonl")
for row in rows:
    print({k: row.get(k) for k in (
        "sequence", "status", "completion_tokens", "duration_s", "ttfc_s",
        "tokens_per_s", "tokens_per_s2_inter_request")})
assert verify_run(run_directory)["status"] == "PASS", "Stop: local replay failed"
print("READBACK_PASS · independent-host validation still NOT_TESTED")

## D. Plot observed request throughput

These are **request-level** rates. Stream chunks are not individual tokenizer tokens.

In [ ]:
import matplotlib.pyplot as plt
x = [r["sequence"]+1 for r in rows]
rates = [r["tokens_per_s"] for r in rows]
acc = [r["tokens_per_s2_inter_request"] for r in rows]
plt.figure(figsize=(9,3.5))
plt.plot(x, rates, marker="o")
plt.title("Akamai vLLM · observed request completion throughput")
plt.xlabel("Request order")
plt.ylabel("Completion tokens/s (wall duration)")
plt.grid(True, alpha=0.3)
plt.show()
plt.figure(figsize=(9,3.5))
plt.plot(x, acc, marker="o")
plt.axhline(0, linestyle="--")
plt.title("Derived inter-request rate change")
plt.xlabel("Request order")
plt.ylabel("Tokens/s² (request midpoint derivative)")
plt.grid(True, alpha=0.3)
plt.show()

## E. Build the self-contained dashboard

For multiple models, rerun the experiment only after the GPU operator has **actually served** another model. Previous run files are preserved; the dashboard combines model comparisons. Claude Lane B results can be imported later as `data/lane_b_results.jsonl` according to the exchange schema.

In [ ]:
from cell_akamai.dashboard import write_dashboard
lane_b = ROOT / "data" / "lane_b_results.jsonl"
dashboard = write_dashboard(
    runs_root=ROOT / "runs",
    lane_b_ref=lane_b if lane_b.is_file() else None,
    output=ROOT / "lane_a_dashboard.html"
)
print("Dashboard:", dashboard)
print("Dashboard SHA-256:", bytes_digest(dashboard.read_bytes()))
print("Open it in JupyterLab's file browser or copy it to a local browser for review.")

## F. Reproducibility receipt

**Interpretation ceiling:** arithmetic and raw file hashes checked locally; actual Akamai-side deployment/config, model weights revision, trusted time, independent-host verification, signature and portfolio MMR require separate receipts.

In [ ]:
summary = json.loads((run_directory / "run_summary.json").read_text())
verification = verify_run(run_directory)
print(json.dumps({
  "run_id": summary["run_id"],
  "endpoint_origin": summary["endpoint_origin"],
  "model": summary["model"],
  "source_class": summary["source_class"],
  "metric_success_count": summary["metric_success_count"],
  "local_readback": verification["status"],
  "model_revision": summary["model_revision"],
  "independent_host": "NOT_TESTED",
  "signed": False,
  "mmr_root": "NOT_COMPUTED",
}, indent=2))